# LLM-as-Judge Evaluation

Two independent LLM judges score chatbot answers on three criteria:
- **Accuracy** — does the answer correctly reflect the source?
- **Completeness** — does it cover all key points from the reference?
- **Faithfulness** — is every claim grounded in the retrieved context?

Inter-judge agreement is measured with Cohen's Kappa.

**Inputs:**
- `evaluation_details/evaluation_gradebook/full_dataset_gradebook.csv`
- `evaluation_details/evaluation_rob2/full_dataset_rob2.csv`

**Output:**
- `evaluation_details/llm_judge/judge_results_grade_{judge_model}.csv`
- `evaluation_details/llm_judge/judge_results_rob2_{judge_model}.csv`

In [ ]:
import os, json, re, warnings
warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

import pandas as pd
import numpy as np
from sklearn.metrics import cohen_kappa_score
from dotenv import load_dotenv
load_dotenv()

from langchain_openai import ChatOpenAI
from rag_core import rag_answer, load_or_build_index, build_llm

## 1. Configuration

In [ ]:
# Judge models (two independent judges)
JUDGE_MODELS = [
    "google/gemini-2.5-flash",
    "deepseek/deepseek-chat-v3-0324",
]

# How many questions to evaluate per domain (set to None to evaluate all)
N_SAMPLE = 40

OUT_DIR = "evaluation_details/llm_judge"
os.makedirs(OUT_DIR, exist_ok=True)

## 2. Load Datasets and Generate Chatbot Answers

In [ ]:
# ---- Load GRADE dataset ----
grade_df = pd.read_csv("evaluation_details/evaluation_gradebook/full_dataset_gradebook.csv")
if N_SAMPLE:
    grade_df = grade_df.sample(min(N_SAMPLE, len(grade_df)), random_state=42).reset_index(drop=True)
print(f"GRADE questions: {len(grade_df)}")

# ---- Load RoB2 dataset ----
rob2_path = "evaluation_details/evaluation_rob2/full_dataset_rob2.csv"
if os.path.exists(rob2_path):
    rob2_df = pd.read_csv(rob2_path)
    if N_SAMPLE:
        rob2_df = rob2_df.sample(min(N_SAMPLE, len(rob2_df)), random_state=42).reset_index(drop=True)
    print(f"RoB2 questions: {len(rob2_df)}")
else:
    print("RoB2 dataset not found — run generate_rob2_dataset.ipynb first.")
    rob2_df = None

In [ ]:
# ---- Generate chatbot answers ----
# Import ask_grade and ask_rob2 from the respective RAG notebooks
# (Run those notebooks first, or copy the index/llm setup here)

from grade_rag import ask_grade, get_grade_contexts  # noqa: requires grade_rag to have been executed

print("Generating GRADE answers...")
grade_df["chatbot_answer"] = grade_df["Question"].apply(ask_grade)
grade_df["retrieved_contexts"] = grade_df["Question"].apply(
    lambda q: " ||| ".join(get_grade_contexts(q, k=5))
)
print("Done.")

In [ ]:
if rob2_df is not None:
    from rob2_rag import ask_rob2, get_rob2_contexts  # noqa

    print("Generating RoB2 answers...")
    rob2_df["chatbot_answer"] = rob2_df["question"].apply(ask_rob2)
    rob2_df["retrieved_contexts"] = rob2_df["question"].apply(
        lambda q: " ||| ".join(get_rob2_contexts(q, k=5))
    )
    print("Done.")

## 3. LLM Judge Prompt

In [ ]:
JUDGE_PROMPT = """\
You are an impartial evaluator assessing the quality of an AI assistant's answer.

Question: {question}

Reference Answer: {reference}

Chatbot Answer: {answer}

Retrieved Context (what the chatbot had available): {context}

Rate the chatbot answer on three criteria, each on a scale of 1-5:

1. **Accuracy** (1=completely wrong, 5=fully correct): Does the answer correctly reflect the source material?
2. **Completeness** (1=major gaps, 5=covers all key points): Does it cover all key points from the reference answer?
3. **Faithfulness** (1=many unsupported claims, 5=fully grounded): Is every claim supported by the retrieved context?

Respond ONLY with a valid JSON object with these exact keys:
{{"accuracy": <int>, "completeness": <int>, "faithfulness": <int>, "justification": "<one sentence>"}}
"""

def judge_answer(question: str, reference: str, answer: str, context: str, judge_llm: ChatOpenAI) -> dict:
    prompt = JUDGE_PROMPT.format(
        question=question,
        reference=reference[:1000],
        answer=answer[:1000],
        context=context[:2000],
    )
    try:
        response = judge_llm.invoke(prompt)
        content = response.content if hasattr(response, 'content') else str(response)
        content = re.sub(r'^```(?:json)?\n?', '', content.strip())
        content = re.sub(r'\n?```$', '', content.strip())
        return json.loads(content)
    except Exception as e:
        return {"accuracy": None, "completeness": None, "faithfulness": None, "justification": str(e)}

## 4. Run Judges on GRADE Dataset

In [ ]:
def run_judge_on_dataset(df: pd.DataFrame, q_col: str, ref_col: str, ans_col: str,
                          ctx_col: str, judge_model: str, domain: str) -> pd.DataFrame:
    judge_llm = ChatOpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=os.getenv("OPENROUTER_API_KEY"),
        model=judge_model,
        temperature=0.0,
    )
    model_slug = judge_model.split("/")[-1]

    results = []
    for i, row in df.iterrows():
        if (i + 1) % 10 == 0:
            print(f"  [{domain}/{model_slug}] {i+1}/{len(df)}")
        scores = judge_answer(
            question=row[q_col],
            reference=str(row[ref_col]),
            answer=str(row[ans_col]),
            context=str(row[ctx_col]),
            judge_llm=judge_llm,
        )
        scores["question"] = row[q_col]
        scores["judge_model"] = judge_model
        results.append(scores)

    result_df = pd.DataFrame(results)
    out_path = f"{OUT_DIR}/judge_results_{domain}_{model_slug}.csv"
    result_df.to_csv(out_path, index=False)
    print(f"Saved: {out_path}")
    return result_df

In [ ]:
grade_judge_results = {}
for judge_model in JUDGE_MODELS:
    print(f"\nRunning judge: {judge_model}")
    res = run_judge_on_dataset(
        df=grade_df,
        q_col="Question", ref_col="Answer", ans_col="chatbot_answer", ctx_col="retrieved_contexts",
        judge_model=judge_model, domain="grade",
    )
    grade_judge_results[judge_model] = res

## 5. Run Judges on RoB2 Dataset

In [ ]:
if rob2_df is not None:
    rob2_judge_results = {}
    for judge_model in JUDGE_MODELS:
        print(f"\nRunning judge: {judge_model}")
        res = run_judge_on_dataset(
            df=rob2_df,
            q_col="question", ref_col="reference_answer", ans_col="chatbot_answer", ctx_col="retrieved_contexts",
            judge_model=judge_model, domain="rob2",
        )
        rob2_judge_results[judge_model] = res

## 6. Inter-Judge Agreement (Cohen's Kappa)

In [ ]:
def compute_kappa(results_dict: dict, domain: str):
    models = list(results_dict.keys())
    if len(models) < 2:
        print("Need at least 2 judges for Kappa.")
        return

    m1, m2 = models[0], models[1]
    r1 = results_dict[m1]
    r2 = results_dict[m2]

    print(f"\n=== Inter-Judge Agreement ({domain}) ===")
    for criterion in ["accuracy", "completeness", "faithfulness"]:
        s1 = r1[criterion].dropna().astype(int)
        s2 = r2[criterion].dropna().astype(int)
        min_len = min(len(s1), len(s2))
        kappa = cohen_kappa_score(s1[:min_len], s2[:min_len])
        print(f"  {criterion}: κ = {kappa:.3f}")

    # Average scores per judge
    for model, res in results_dict.items():
        slug = model.split("/")[-1]
        print(f"\n  {slug} averages:")
        for c in ["accuracy", "completeness", "faithfulness"]:
            print(f"    {c}: {res[c].mean():.2f}")

compute_kappa(grade_judge_results, "GRADE")
if rob2_df is not None:
    compute_kappa(rob2_judge_results, "RoB2")

## 7. Combined Summary

In [ ]:
summary_rows = []
for domain, results_dict in [("GRADE", grade_judge_results)] + \
    ([("RoB2", rob2_judge_results)] if rob2_df is not None else []):
    for judge_model, res in results_dict.items():
        row = {
            "domain": domain,
            "judge_model": judge_model,
            "n": len(res),
            "accuracy_mean": res["accuracy"].mean(),
            "completeness_mean": res["completeness"].mean(),
            "faithfulness_mean": res["faithfulness"].mean(),
        }
        summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(f"{OUT_DIR}/judge_summary.csv", index=False)
summary_df